In [2]:
import json
from collections import defaultdict

In [3]:

def analyze_moderation_jsonl(file_path: str) -> dict:
    """
    Reads a large JSONL file line-by-line and generates comprehensive analytics
    comparing 'prompt_label' (ground truth) vs model 'flagged' state (prediction).
    """
    # Confusion Matrix Counters
    # Ground Truth: unsafe (Positive), safe (Negative)
    tp = 0  # True Positive:  actual unsafe, predicted flagged
    fp = 0  # False Positive: actual safe, predicted flagged
    tn = 0  # True Negative:  actual safe, predicted unflagged
    fn = 0  # False Negative: actual unsafe, predicted unflagged
    
    total_records = 0
    invalid_records = 0
    
    # Category statistics
    category_counts = defaultdict(int)
    category_scores = defaultdict(list)
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                invalid_records += 1
                continue
            
            total_records += 1
            
            # Ground Truth
            actual_unsafe = (record.get("prompt_label") == "unsafe")
            
            # Prediction from OpenAI/Gemini format
            results = record.get("results", [])
            predicted_flagged = False
            
            if results and isinstance(results, list):
                res = results[0]
                predicted_flagged = res.get("flagged", False)
                
                # Aggregate category breakdowns if flagged
                categories = res.get("categories", {})
                scores = res.get("category_scores", {})
                
                for cat, is_active in categories.items():
                    if is_active:
                        category_counts[cat] += 1
                        if cat in scores:
                            category_scores[cat].append(scores[cat])
            
            # Update Confusion Matrix
            if actual_unsafe and predicted_flagged:
                tp += 1
            elif not actual_unsafe and predicted_flagged:
                fp += 1
            elif not actual_unsafe and not predicted_flagged:
                tn += 1
            elif actual_unsafe and not predicted_flagged:
                fn += 1

    # Safe metric calculations (avoid division by zero)
    accuracy = (tp + tn) / total_records if total_records > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0.0

    # Category summaries
    category_analytics = {}
    for cat, count in category_counts.items():
        scores_list = category_scores[cat]
        avg_score = sum(scores_list) / len(scores_list) if scores_list else 0.0
        category_analytics[cat] = {
            "flagged_count": count,
            "avg_confidence_score": round(avg_score, 4)
        }

    stats = {
        "dataset_summary": {
            "total_records": total_records,
            "invalid_records": invalid_records,
            "actual_unsafe_count": tp + fn,
            "actual_safe_count": fp + tn,
            "predicted_flagged_count": tp + fp,
            "predicted_unflagged_count": tn + fn,
        },
        "confusion_matrix": {
            "true_positives": tp,
            "false_positives": fp,
            "true_negatives": tn,
            "false_negatives": fn
        },
        "metrics": {
            "accuracy": round(accuracy, 4),
            "precision": round(precision, 4),
            "recall_sensitivity": round(recall, 4),
            "f1_score": round(f1_score, 4),
            "false_positive_rate": round(fpr, 4),
            "false_negative_rate": round(fnr, 4),
        },
        "category_analytics": category_analytics
    }
    
    return stats


def print_report(stats: dict) -> None:
    """Prints a formatted summary of the moderation stats."""
    ds = stats["dataset_summary"]
    cm = stats["confusion_matrix"]
    m = stats["metrics"]

    print("=" * 50)
    print("      CONTENT MODERATION EVALUATION REPORT      ")
    print("=" * 50)
    print(f"Total Processed Records : {ds['total_records']}")
    print(f"Actual Unsafe (Pos)     : {ds['actual_unsafe_count']}")
    print(f"Actual Safe (Neg)       : {ds['actual_safe_count']}\n")
    
    print("CONFUSION MATRIX:")
    print(f"  TP: {cm['true_positives']:<6} | FP: {cm['false_positives']}")
    print(f"  FN: {cm['false_negatives']:<6} | TN: {cm['true_negatives']}\n")

    print("PERFORMANCE METRICS:")
    print(f"  Accuracy            : {m['accuracy'] * 100:.2f}%")
    print(f"  Precision           : {m['precision'] * 100:.2f}%")
    print(f"  Recall (Sensitivity): {m['recall_sensitivity'] * 100:.2f}%")
    print(f"  F1-Score            : {m['f1_score'] * 100:.2f}%")
    print(f"  False Positive Rate : {m['false_positive_rate'] * 100:.2f}%")
    print(f"  False Negative Rate : {m['false_negative_rate'] * 100:.2f}%\n")

    print("FLAGGED CATEGORY BREAKDOWN:")
    for cat, info in stats["category_analytics"].items():
        print(f"  - {cat:<22}: {info['flagged_count']} flags (Avg Score: {info['avg_confidence_score']})")
    print("=" * 50)

In [ ]:
input_file_1 = "C:\\Users\\Milind\\MyData\\git\\datasets\\content-moderation-dataset\\Aegis-AI-Content-Safety-Dataset-2.0\\merged-output\\test.json-out.jsonl"
input_file = "C:\\Users\\Milind\\MyData\\git\\datasets\\content-moderation-dataset\\Aegis-AI-Content-Safety-Dataset-2.0\\merged-output\\train.json-out.jsonl"


stats = analyze_moderation_jsonl(input_file)
print_report(stats)

      CONTENT MODERATION EVALUATION REPORT      
Total Processed Records : 1964
Actual Unsafe (Pos)     : 1059
Actual Safe (Neg)       : 905

CONFUSION MATRIX:
  TP: 864    | FP: 116
  FN: 195    | TN: 789

PERFORMANCE METRICS:
  Accuracy            : 84.16%
  Precision           : 88.16%
  Recall (Sensitivity): 81.59%
  F1-Score            : 84.75%
  False Positive Rate : 12.82%
  False Negative Rate : 18.41%

FLAGGED CATEGORY BREAKDOWN:
  - harassment            : 274 flags (Avg Score: 0.9378)
  - violence              : 604 flags (Avg Score: 0.9642)
  - sexual                : 43 flags (Avg Score: 0.9723)
  - hate                  : 154 flags (Avg Score: 0.9494)
